# Proactive Retail Intelligence with Snowflake Cortex

A multi-tenant retail analytics SaaS provider wants its embedded assistant to stop waiting for questions and start **leading** with explained observations — without paying a language model to scan the whole data estate. This lab builds that proactive layer entirely inside Snowflake.

The pattern: **cheap in-warehouse ML detects and explains; the LLM only narrates the small flagged slice.**

### What You'll Learn

- Detect anomalies across every store with `SNOWFLAKE.ML.ANOMALY_DETECTION` — no LLM, no tokens.
- Explain the *why* with `SNOWFLAKE.ML.TOP_INSIGHTS` (key-driver analysis).
- Answer ad-hoc questions on live data through a semantic view + Cortex Analyst.
- Build a custom **briefing tool** and an **optimized Cortex Agent** that orchestrates every layer.
- Surface the same agent in **Snowflake CoWork** and call it over REST `agent:run`.

**Prerequisite:** Run `lab/setup.sql` first. It fully populates the data and builds the ML models, findings/drivers tables, semantic view, and Cortex Search service.

---
## Section 1: Connect & Explore

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE PROACTIVE_RETAIL_DEMO").collect()
session.sql("USE SCHEMA ANALYTICS").collect()
session.sql("USE WAREHOUSE PROACTIVE_RETAIL_WH").collect()
print("Connected to PROACTIVE_RETAIL_DEMO.ANALYTICS")

In [ ]:
# Tour the raw data: tenants, stores, daily metrics, and individual returns
session.sql("""
    SELECT 'retailers (tenants)' AS object, COUNT(*) AS row_count FROM RAW.RETAILERS
    UNION ALL SELECT 'stores',              COUNT(*) FROM RAW.STORES
    UNION ALL SELECT 'store_day_metrics',   COUNT(*) FROM RAW.STORE_DAY_METRICS
    UNION ALL SELECT 'returns_fact',         COUNT(*) FROM RAW.RETURNS_FACT
    ORDER BY row_count DESC
""").show()

---
## Section 2: Detect Anomalies Without an LLM

`setup.sql` already trained `RETURN_RATE_DETECTOR` (a multi-series `SNOWFLAKE.ML.ANOMALY_DETECTION` model — one return-rate series per store) and wrote the flagged rows to `STORE_ANOMALY_FINDINGS`. The `STORE_ANOMALIES_ENRICHED` view joins them to store context.

This is the proactive engine: a scheduled `TASK` keeps the findings fresh. **No language model was involved in finding these.**

In [ ]:
# The stores the in-warehouse scan flagged (highest deviation first)
session.sql("""
    SELECT store_id, region, metro, store_format,
           metric_date::DATE       AS on_date,
           ROUND(return_rate, 3)    AS actual_rate,
           ROUND(expected_return_rate, 3) AS expected_rate,
           ROUND(distance, 2)       AS anomaly_distance
    FROM ANALYTICS.STORE_ANOMALIES_ENRICHED
    ORDER BY anomaly_distance DESC
    LIMIT 20
""").show()

---
## Section 3: Explain the "Why" with Top Insights

Detection tells you *that* return rates moved. `SNOWFLAKE.ML.TOP_INSIGHTS` tells you *why*. `setup.sql` labeled the recent 10-day window as the test group (vs. the prior baseline), set the metric to refunded dollars, ran `GET_DRIVERS`, and persisted the ranked drivers to `RETURN_DRIVERS`.

The drivers are the raw material for a sentence like *"concentrated in Electronics across three Midwest stores, trending since last week."*

In [ ]:
# The ranked drivers behind the recent jump in refunded dollars.
# Columns are produced by GET_DRIVERS; SELECT * so we don't assume the schema.
session.sql("""
    SELECT * FROM ANALYTICS.RETURN_DRIVERS
    LIMIT 25
""").show()

---
## Section 4: Ad-Hoc Analysis on Live Data

The proactive briefing is half the story — managers also ask follow-ups. The `RETAIL_RETURNS_SV` semantic view exposes the same tables as a governed model that Cortex Analyst can query in natural language. Below is the kind of SQL Analyst compiles for *"what should I focus on at Store 1042?"* — every number comes from this account's live data.

In [ ]:
# Refund dollars and return counts by category at an anomaly store, via the semantic view.
session.sql("""
    SELECT * FROM SEMANTIC_VIEW(
        ANALYTICS.RETAIL_RETURNS_SV
        DIMENSIONS returns.product_category, stores.store_id_dim
        METRICS   returns.total_refund, returns.return_txn_count
    )
    WHERE store_id_dim = 1042
    ORDER BY total_refund DESC
""").show()

---
## Section 5: Build the Custom "Briefing" Tool

The agent needs a cheap way to answer *"what changed?"* without re-running ML on every turn. We wrap the precomputed findings + drivers in a SQL function that returns a **single JSON cell** (the required shape for a `generic` agent tool). The agent calls this once and narrates the result — the language model never touches the raw tables.

In [ ]:
# A generic agent tool must return ONE cell. We return a JSON string that bundles
# the top flagged stores and the ranked drivers the ML layer already computed.
session.sql("""
    CREATE OR REPLACE FUNCTION ANALYTICS.GET_PROACTIVE_BRIEFING()
    RETURNS VARCHAR
    COMMENT = 'Returns a JSON briefing of the currently flagged store anomalies and their top drivers.'
    AS
    $$
      TO_JSON(OBJECT_CONSTRUCT(
        'generated_at', CURRENT_TIMESTAMP()::STRING,
        'flagged_stores', (
            SELECT ARRAY_AGG(OBJECT_CONSTRUCT(
                'store_id', store_id, 'metro', metro, 'region', region,
                'actual_return_rate', ROUND(return_rate, 3),
                'expected_return_rate', ROUND(expected_return_rate, 3)))
            FROM (
                SELECT store_id, metro, region, return_rate, expected_return_rate
                FROM ANALYTICS.STORE_ANOMALIES_ENRICHED
                ORDER BY distance DESC
                LIMIT 10)),
        'top_drivers', (
            SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
            FROM (SELECT * FROM ANALYTICS.RETURN_DRIVERS LIMIT 15))
      ))
    $$
""").collect()

# Grant the agent's role access, then preview the single-cell JSON the agent will read.
session.sql("GRANT USAGE ON FUNCTION ANALYTICS.GET_PROACTIVE_BRIEFING() TO ROLE SYSADMIN").collect()
session.sql("SELECT ANALYTICS.GET_PROACTIVE_BRIEFING() AS briefing_json").show()

---
## Section 6: Create the Cortex Agent (baseline)

One agent orchestrates every layer: the briefing tool for *"what changed"*, Cortex Analyst for store metrics, Cortex Search for reason text, and `data_to_chart` for visuals. We start with a **baseline** spec — thin tool descriptions, no orchestration guidance — so the optimization in Section 6b has something to improve.

> After creating the agent, bind the custom tool in Snowsight → **AI & ML → Agents → (this agent) → Tools**: add a custom tool of resource type *Function*, identifier `PROACTIVE_RETAIL_DEMO.ANALYTICS.GET_PROACTIVE_BRIEFING`, warehouse `PROACTIVE_RETAIL_WH`. Custom-tool binding is done in the UI.

In [ ]:
# BASELINE agent — intentionally thin descriptions and no orchestration routing.
session.sql("""
    CREATE OR REPLACE AGENT ANALYTICS.PROACTIVE_RETAIL_AGENT_BASELINE
    COMMENT = 'Baseline retail assistant (pre-optimization).'
    PROFILE = '{"display_name": "Retail Assistant (Baseline)"}'
    FROM SPECIFICATION
    $$
    models:
      orchestration: auto
    instructions:
      response: "Answer questions about retail returns."
    tools:
      - tool_spec:
          type: "generic"
          name: "get_proactive_briefing"
          description: "Returns a briefing."
          input_schema:
            type: object
            properties: {}
      - tool_spec:
          type: "cortex_analyst_text_to_sql"
          name: "store_analyst"
          description: "Store data."
      - tool_spec:
          type: "cortex_search"
          name: "reason_search"
          description: "Return reasons."
    tool_resources:
      store_analyst:
        semantic_view: "PROACTIVE_RETAIL_DEMO.ANALYTICS.RETAIL_RETURNS_SV"
        execution_environment:
          type: warehouse
          warehouse: PROACTIVE_RETAIL_WH
      reason_search:
        name: "PROACTIVE_RETAIL_DEMO.ANALYTICS.RETURN_REASONS_SEARCH"
        id_column: "RETURN_ID"
        max_results: "5"
    $$
""").collect()
print("Baseline agent created.")

---
## Section 6b: Optimize the Agent (required)

Agent quality lives in the tool descriptions and orchestration instructions — they are how the orchestrator decides *which* tool to call. The optimized spec below:

- **Names + describes each tool by domain and function**, and says *when NOT to use it* (the single biggest driver of correct tool selection).
- **Adds an orchestration instruction** that routes "what changed / anything I should know" to the briefing tool first, store metrics to Analyst, and reason text to Search.
- **Adds a response instruction** (lead with the headline, then the driver, then next steps) and **sample questions** that seed the proactive experience.

The before/after specs are saved under `agent_optimization/versions/` with a diff summary in `agent_optimization/README.md`.

In [ ]:
# OPTIMIZED agent — purpose-driven tool descriptions + orchestration + response guidance.
session.sql("""
    CREATE OR REPLACE AGENT ANALYTICS.PROACTIVE_RETAIL_AGENT
    COMMENT = 'Proactive retail intelligence assistant (optimized).'
    PROFILE = '{"display_name": "Proactive Retail Assistant", "color": "blue"}'
    FROM SPECIFICATION
    $$
    models:
      orchestration: auto
    orchestration:
      budget:
        seconds: 30
        tokens: 16000
    instructions:
      response: "Lead with the single most important change (the headline), then the driver (which category / stores / metro, since when), then 2-3 concrete next steps. Be concise; do not restate raw JSON."
      orchestration: "For 'what changed', 'anything I should know', or opening a session, call get_proactive_briefing FIRST and narrate it. For questions about a specific store's metrics, refund dollars, return rates, or trends, use store_analyst. For the free-text reasons customers gave, use reason_search. Use data_to_chart when the user asks to visualize or compare."
      sample_questions:
        - question: "What changed at my stores overnight?"
        - question: "I'm visiting Store 1042 — what should I focus on?"
        - question: "Why are returns up in Electronics?"
    tools:
      - tool_spec:
          type: "generic"
          name: "get_proactive_briefing"
          description: "Returns a JSON briefing of the CURRENTLY FLAGGED store return-rate anomalies and their top drivers (product category, stores, metro), precomputed by in-warehouse ML. Use this to lead a session or answer 'what changed / anything I should know'. Do NOT use it for a specific metric value or a single named store's history — use store_analyst for that."
          input_schema:
            type: object
            properties: {}
      - tool_spec:
          type: "cortex_analyst_text_to_sql"
          name: "store_analyst"
          description: "Answers quantitative questions about stores, returns, refunds, and daily metrics from the RETAIL_RETURNS_SV semantic view (governed SQL over live data). Use for refund dollars, return counts, return rates, sales, and trends by store / region / metro / category / date. Do NOT use for free-text return reasons (use reason_search) or for the current anomaly list (use get_proactive_briefing)."
      - tool_spec:
          type: "cortex_search"
          name: "reason_search"
          description: "Semantic search over the free-text explanations customers gave when returning items. Use to find WHY customers say they returned things (defects, sizing, damage, descriptions). Do NOT use for numeric aggregates — use store_analyst."
      - tool_spec:
          type: "data_to_chart"
          name: "data_to_chart"
          description: "Generates a chart from data returned by another tool. Use when the user asks to visualize, plot, or compare."
    tool_resources:
      store_analyst:
        semantic_view: "PROACTIVE_RETAIL_DEMO.ANALYTICS.RETAIL_RETURNS_SV"
        execution_environment:
          type: warehouse
          warehouse: PROACTIVE_RETAIL_WH
      reason_search:
        name: "PROACTIVE_RETAIL_DEMO.ANALYTICS.RETURN_REASONS_SEARCH"
        id_column: "RETURN_ID"
        max_results: "5"
    $$
""").collect()
print("Optimized agent PROACTIVE_RETAIL_AGENT created. Bind the custom tool in the Agents UI, then chat with it.")

---
## Section 7: Deliver the Agent — Snowflake CoWork

The same `PROACTIVE_RETAIL_AGENT` object now serves two surfaces from one definition:

**1. Interactive (Snowflake CoWork).** Go to Snowsight → **AI & ML → Agents**, open **Proactive Retail Assistant**, and chat. Try the seeded questions:
- *"What changed at my stores overnight?"* → the agent calls `get_proactive_briefing` and leads with the headline + driver + next steps.
- *"I'm visiting Store 1042 — what should I focus on?"* → routes to Cortex Analyst over the semantic view.
- *"Why are returns up in Electronics?"* → blends Search (reasons) with Analyst (magnitude).

**2. Embedded (REST `agent:run`).** The product's in-app assistant calls the identical agent and streams the answer into its own chat surface:

```
POST /api/v2/databases/PROACTIVE_RETAIL_DEMO/schemas/ANALYTICS/agents/PROACTIVE_RETAIL_AGENT:run
Authorization: Bearer <PAT or OAuth token>
{
  "messages": [
    { "role": "user",
      "content": [ { "type": "text", "text": "What changed at my stores overnight?" } ] }
  ]
}
```

One agent, one governance model, two delivery paths — and the language model only ever narrates the small flagged slice.

---
## Summary

You built a proactive intelligence layer end to end, entirely in Snowflake:

- **Detected** anomalies across every store with `SNOWFLAKE.ML.ANOMALY_DETECTION` — no LLM, no tokens — and saw how a scheduled `TASK` keeps the findings fresh.
- **Explained** the change with `SNOWFLAKE.ML.TOP_INSIGHTS` key-driver analysis.
- **Answered ad-hoc** questions on live data through the `RETAIL_RETURNS_SV` semantic view + Cortex Analyst.
- **Wrapped** the precomputed findings + drivers in a single-cell briefing tool.
- **Built and optimized** a Cortex Agent that orchestrates the briefing tool, Analyst, Search, and charts — then surfaced it in Snowflake CoWork and via REST `agent:run`.

The takeaway: **detect and explain cheaply in the warehouse; let the language model narrate and converse over the small slice that already matters.** To reset, run `lab/cleanup.sql`.